## Project Overview: FIFA World Cup 2026 Prediction

This project aims to predict the outcomes of the FIFA World Cup 2026 matches, including group stage and knockout rounds. The predictions cover match scores, corners, and cards, with a focus on accuracy to align with the competition's scoring system.

The methodology involves:

1.  **Data Ingestion and Preparation**: Loading group stage fixtures, knockout slots, FIFA rankings, and historical match results. Data is cleaned by removing unplayed matches and filtering for recent non-friendly matches (from 2018 onwards).
2.  **Team Strength Modeling**: A Poisson regression-based model is used to estimate attack and defense ratings for each team, incorporating a home-field advantage factor. This model is fitted using historical match data.
3.  **Group Stage Prediction**: Utilizing the team strength model, individual match scorelines and winners are predicted. A Monte Carlo simulation (10,000 iterations) is then run to determine the most likely group winners, runners-up, and best third-place teams.
4.  **Knockout Stage Prediction**: The predicted group stage outcomes are used to populate the knockout bracket. Each knockout match is simulated using the team strength model to predict scores and winners, propagating the results through subsequent rounds.
5.  **Validation**: A backtesting framework is implemented using past World Cup matches (pre-2018) to evaluate the model's performance based on the competition's scoring rules for exact scores and correct winners.

The project demonstrates a comprehensive approach to soccer match prediction, combining statistical modeling with simulation to generate robust forecasts for a major sporting event.

# ⚽ Predict the FIFA World Cup 2026

## 📖 Background

The 2026 FIFA World Cup is one of the biggest sporting events in the world, hosted across the United States, Canada, and Mexico. For the first time, the tournament expands to 48 teams, producing 104 matches across the group stage and knockout rounds.

Using machine learning, historical statistics, and soccer domain knowledge, predict match scores, corners, and cards for every fixture. You must submit all your predictions before a single ball is kicked.

The scoring system rewards precision: an exact scoreline earns maximum points, while close predictions still earn partial credit. Later rounds carry score multipliers, so a strong model that holds up in the knockout stages can leapfrog the competition. The challenge is designed to be difficult enough that no one can achieve a perfect score—even with AI assistance—but accessible enough that any data enthusiast can participate and score points.

## 💾 The data

You have access to the following files:

#### `data/group_fixtures.csv` — all 72 group stage matches
| Variable | Description |
|---|---|
| `match_id` | Unique match identifier |
| `group` | Group letter (A–L) |
| `home_team` | Home team name |
| `away_team` | Away team name |
| `date` | Match date (UTC) |
| `venue` | Stadium and city |

#### `data/knockout_slots.csv` — all 32 knockout round slots
| Variable | Description |
|---|---|
| `match_id` | Unique match identifier |
| `round` | Round name (e.g. `Quarter-final`) |
| `multiplier` | Score multiplier for this round |
| `slot_home` | Description of the home team slot (e.g. `Winner Group A`) |
| `slot_away` | Description of the away team slot |

| Variable | Description |
|---|---|

You may also bring in any external data—FIFA rankings, historical match results, player statistics—to build your predictions.

In [ ]:
import pandas as pd

group_fixtures = pd.read_csv('group_fixtures.csv')
group_fixtures.head()

,match_id,group,home_team,away_team,date_utc,venue
0,1,A,Mexico,South Africa,2026-06-11T19:00:00Z,"Estadio Azteca, Mexico City"
1,2,A,South Korea,UEFA Playoff D,2026-06-12T02:00:00Z,"Estadio Akron, Guadalajara"
2,3,B,Canada,UEFA Playoff A,2026-06-12T19:00:00Z,"BMO Field, Toronto"
3,4,D,USA,Paraguay,2026-06-13T01:00:00Z,"SoFi Stadium, Los Angeles"
4,5,D,Australia,UEFA Playoff C,2026-06-13T04:00:00Z,"BC Place, Vancouver"


In [ ]:
knockout_slots = pd.read_csv('knockout_slots.csv')
knockout_slots

,match_id,round,multiplier,date_utc,venue,slot_home,slot_away
0,73,Round of 32,1,2026-06-28T19:00:00Z,"SoFi Stadium, Los Angeles",Runner-up Group A,Runner-up Group B
1,74,Round of 32,1,2026-06-29T17:00:00Z,"NRG Stadium, Houston",Winner Group C,Runner-up Group F
2,75,Round of 32,1,2026-06-29T20:30:00Z,"Gillette Stadium, Boston",Winner Group E,Best 3rd (Groups A/B/C/D/F)
3,76,Round of 32,1,2026-06-30T01:00:00Z,"Estadio BBVA, Monterrey",Winner Group F,Runner-up Group C
4,77,Round of 32,1,2026-06-30T17:00:00Z,"AT&T Stadium, Dallas",Runner-up Group E,Runner-up Group I
5,78,Round of 32,1,2026-06-30T21:00:00Z,"MetLife Stadium, East Rutherford",Winner Group I,Best 3rd (Groups C/D/F/G/H)
6,79,Round of 32,1,2026-07-01T01:00:00Z,"Estadio Azteca, Mexico City",Winner Group A,Best 3rd (Groups C/E/F/H/I)
7,80,Round of 32,1,2026-07-01T16:00:00Z,"Mercedes-Benz Stadium, Atlanta",Winner Group L,Best 3rd (Groups E/H/I/J/K)
8,81,Round of 32,1,2026-07-01T20:00:00Z,"Lumen Field, Seattle",Winner Group G,Best 3rd (Groups A/E/H/I/J)
9,82,Round of 32,1,2026-07-02T00:00:00Z,"Levi's Stadium, Santa Clara",Winner Group D,Best 3rd (Groups B/E/F/I/J)


## 🗓️ Group stage predictions

Fill in your predictions for all 72 group stage matches below.

In [ ]:
group_predictions = group_fixtures.copy()

# Fill in your predictions for each match
# Example (match 1 — Mexico vs South Africa): predicted_home_goals=2, predicted_away_goals=1, corners=9, yellow_cards=3, red_cards=0, winning_team='home'
group_predictions['predicted_home_goals'] = None   # e.g. 2
group_predictions['predicted_away_goals'] = None   # e.g. 1
group_predictions['corners']              = None   # e.g. 9
group_predictions['yellow_cards']         = None   # e.g. 3
group_predictions['red_cards']            = None   # e.g. 0
group_predictions['winning_team']         = None   # "home", "away", or "draw"

group_predictions

,match_id,group,home_team,away_team,date_utc,venue,predicted_home_goals,predicted_away_goals,corners,yellow_cards,red_cards,winning_team
0,1,A,Mexico,South Africa,2026-06-11T19:00:00Z,"Estadio Azteca, Mexico City",None,None,None,None,None,None
1,2,A,South Korea,UEFA Playoff D,2026-06-12T02:00:00Z,"Estadio Akron, Guadalajara",None,None,None,None,None,None
2,3,B,Canada,UEFA Playoff A,2026-06-12T19:00:00Z,"BMO Field, Toronto",None,None,None,None,None,None
3,4,D,USA,Paraguay,2026-06-13T01:00:00Z,"SoFi Stadium, Los Angeles",None,None,None,None,None,None
4,5,D,Australia,UEFA Playoff C,2026-06-13T04:00:00Z,"BC Place, Vancouver",None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...
67,68,L,Croatia,Ghana,2026-06-27T21:00:00Z,"Lincoln Financial Field, Philadelphia",None,None,None,None,None,None
68,69,K,Colombia,Portugal,2026-06-27T23:30:00Z,"Hard Rock Stadium, Miami",None,None,None,None,None,None
69,70,K,FIFA Playoff 1,Uzbekistan,2026-06-27T23:30:00Z,"Mercedes-Benz Stadium, Atlanta",None,None,None,None,None,None
70,71,J,Algeria,Austria,2026-06-28T02:00:00Z,"GEHA Field at Arrowhead Stadium, Kansas City",None,None,None,None,None,None


### Importing external data

Import FIFA Rankings External data

In [ ]:
fifa_rankings = pd.read_csv('fifa_ranking.csv', index_col="Rank")
fifa_rankings.head()

,Team,Latest results,Upcoming matches,Points
Rank,,,,
1,France,FIFA World Cup 26™ UEFA QualifiersGroup D13 Oc...,NaN,1877.32
2,Spain,FIFA World Cup 26™ UEFA QualifiersGroup E14 Oc...,SpainIraq04 Jun19:00,1876.40
3,Argentina,Friendly 202511 October 2025Argentina1Venezuel...,ArgentinaHonduras07 Jun00:00,1874.81
4,England,FIFA World Cup 26™ UEFA QualifiersGroup K14 Oc...,NaN,1825.97
5,Portugal,FIFA World Cup 26™ UEFA QualifiersGroup F14 Oc...,NaN,1763.83


Import International Results from 1872 - 2023

In [ ]:
results = pd.read_csv('results.csv', encoding='latin1')
print(results.head())
print(results.shape)

         date home_team away_team  home_score  away_score tournament     city  \
0  1872-11-30  Scotland   England         0.0         0.0   Friendly  Glasgow   
1  1873-03-08   England  Scotland         4.0         2.0   Friendly   London   
2  1874-03-07  Scotland   England         2.0         1.0   Friendly  Glasgow   
3  1875-03-06   England  Scotland         2.0         2.0   Friendly   London   
4  1876-03-04  Scotland   England         3.0         0.0   Friendly  Glasgow   

    country  id  
0  Scotland   1  
1   England   2  
2  Scotland   3  
3   England   4  
4  Scotland   5  
(43281, 9)


Remove the rows with no results (future or unplayed matches)

In [ ]:
# Remove rows with no scores (future or unplayed matches)
results = results.dropna(subset=['home_score', 'away_score'])
print(results.shape)

(43277, 9)


Filter for the most recent matches (from 2018)

In [ ]:
# Filter results to relevant matches only
results['date'] = pd.to_datetime(results['date'])

# Keep last ~8 years, drop pure friendlies
recent = results[
    (results['date'] >= '2018-01-01') &
    (results['tournament'] != 'Friendly')
].copy()

In [ ]:
recent.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,id
38233,2018-01-02,Iraq,United Arab Emirates,0.0,0.0,Gulf Cup,Kuwait City,Kuwait,38234
38234,2018-01-02,Oman,Bahrain,1.0,0.0,Gulf Cup,Kuwait City,Kuwait,38235
38235,2018-01-05,Oman,United Arab Emirates,0.0,0.0,Gulf Cup,Kuwait City,Kuwait,38236
38240,2018-01-13,Morocco,Mauritania,4.0,0.0,African Nations Championship,Casablanca,Morocco,38241
38242,2018-01-14,Guinea,Sudan,1.0,2.0,African Nations Championship,Casablanca,Morocco,38243


In [ ]:
recent.shape

(3489, 9)

Team strength model: Fit Poisson attack/defense ratings

In [ ]:
from scipy.optimize import minimize
import numpy as np

teams = pd.unique(recent[['home_team', 'away_team']].values.ravel())
team_idx = {t: i for i, t in enumerate(teams)}
n = len(teams)
HOME_ADV = 0.3

In [ ]:
# Pre-compute index arrays
home_idx   = recent['home_team'].map(team_idx).values
away_idx   = recent['away_team'].map(team_idx).values
home_goals = recent['home_score'].values
away_goals = recent['away_score'].values

valid = ~(pd.isna(recent['home_team'].map(team_idx)) | pd.isna(recent['away_team'].map(team_idx)))
home_idx   = home_idx[valid].astype(int)
away_idx   = away_idx[valid].astype(int)
home_goals = home_goals[valid]
away_goals = away_goals[valid]

def log_likelihood(params):
    attack  = params[:n]
    defense = params[n:2*n]
    home    = params[2*n]
    mu_h = np.exp(attack[home_idx] - defense[away_idx] + home)
    mu_a = np.exp(attack[away_idx] - defense[home_idx])
    return -(home_goals * np.log(mu_h) - mu_h + away_goals * np.log(mu_a) - mu_a).sum()

x0 = np.zeros(2*n + 1)
x0[2*n] = HOME_ADV
res = minimize(log_likelihood, x0, method='L-BFGS-B')

# these 3 lines were missing
attack_ratings  = dict(zip(teams, res.x[:n]))
defense_ratings = dict(zip(teams, res.x[n:2*n]))
home_advantage  = res.x[2*n]

In [ ]:
playoff_map = {
    'UEFA Playoff A': 'Ukraine',
    'UEFA Playoff B': 'Greece',
    'UEFA Playoff C': 'Romania',
    'UEFA Playoff D': 'Slovenia',
    'FIFA Playoff 1': 'Indonesia',
    'FIFA Playoff 2': 'Venezuela',
}

group_fixtures['home_team'] = group_fixtures['home_team'].replace(playoff_map)
group_fixtures['away_team'] = group_fixtures['away_team'].replace(playoff_map)

Group Stage Predictions

In [ ]:
# Compute expected goals per match
def expected_goals(home_team, away_team):
    mu_h = np.exp(attack_ratings.get(home_team, 0) - defense_ratings.get(away_team, 0) + home_advantage)
    mu_a = np.exp(attack_ratings.get(away_team, 0) - defense_ratings.get(home_team, 0))
    return mu_h, mu_a

In [ ]:
# Pick the best scoreline
from scipy.stats import poisson

def predict_scoreline(mu_h, mu_a, max_goals=8):
    home_probs = poisson.pmf(np.arange(max_goals+1), mu_h)
    away_probs = poisson.pmf(np.arange(max_goals+1), mu_a)
    matrix = np.outer(home_probs, away_probs)  # shape (9,9)
    idx = np.unravel_index(matrix.argmax(), matrix.shape)
    return idx[0], idx[1]  # predicted home goals, away goals

In [ ]:
# Derive win probability and set winning_team
def win_probabilities(mu_h, mu_a, max_goals=8):
    home_probs = poisson.pmf(np.arange(max_goals+1), mu_h)
    away_probs = poisson.pmf(np.arange(max_goals+1), mu_a)
    matrix = np.outer(home_probs, away_probs)
    p_home = np.tril(matrix, -1).sum()   # home goals > away goals
    p_away = np.triu(matrix, 1).sum()    # away goals > home goals
    p_draw = np.trace(matrix)
    return p_home, p_draw, p_away
def pick_winner(p_home, p_draw, p_away):
    return ['home', 'draw', 'away'][np.argmax([p_home, p_draw, p_away])]

In [ ]:
# Fill group_predictions
for i, row in group_fixtures.iterrows():
    mu_h, mu_a = expected_goals(row['home_team'], row['away_team'])
    h_goals, a_goals = predict_scoreline(mu_h, mu_a)
    p_home, p_draw, p_away = win_probabilities(mu_h, mu_a)

    group_predictions.loc[i, 'predicted_home_goals'] = h_goals
    group_predictions.loc[i, 'predicted_away_goals'] = a_goals
    group_predictions.loc[i, 'winning_team']         = pick_winner(p_home, p_draw, p_away)
    group_predictions.loc[i, 'corners']              = 10   # baseline
    group_predictions.loc[i, 'yellow_cards']         = 3    # baseline
    group_predictions.loc[i, 'red_cards']            = 0    # baseline

In [ ]:
group_predictions

,match_id,group,home_team,away_team,date_utc,venue,predicted_home_goals,predicted_away_goals,corners,yellow_cards,red_cards,winning_team
0,1,A,Mexico,South Africa,2026-06-11T19:00:00Z,"Estadio Azteca, Mexico City",2,0,10,3,0,home
1,2,A,South Korea,UEFA Playoff D,2026-06-12T02:00:00Z,"Estadio Akron, Guadalajara",0,1,10,3,0,away
2,3,B,Canada,UEFA Playoff A,2026-06-12T19:00:00Z,"BMO Field, Toronto",2,0,10,3,0,home
3,4,D,USA,Paraguay,2026-06-13T01:00:00Z,"SoFi Stadium, Los Angeles",1,0,10,3,0,home
4,5,D,Australia,UEFA Playoff C,2026-06-13T04:00:00Z,"BC Place, Vancouver",1,0,10,3,0,home
...,...,...,...,...,...,...,...,...,...,...,...,...
67,68,L,Croatia,Ghana,2026-06-27T21:00:00Z,"Lincoln Financial Field, Philadelphia",1,0,10,3,0,home
68,69,K,Colombia,Portugal,2026-06-27T23:30:00Z,"Hard Rock Stadium, Miami",1,0,10,3,0,home
69,70,K,FIFA Playoff 1,Uzbekistan,2026-06-27T23:30:00Z,"Mercedes-Benz Stadium, Atlanta",0,3,10,3,0,away
70,71,J,Algeria,Austria,2026-06-28T02:00:00Z,"GEHA Field at Arrowhead Stadium, Kansas City",1,0,10,3,0,home


Simulate Group Standings

Simulate groups: Monte Carlo to get group winners/runners-up + best-3rd ranking → fill knockout matchups.

In [ ]:
# Simulate one tournament (single group stage run)
from scipy.stats import poisson
import numpy as np

def simulate_group_stage(group_fixtures, attack_ratings, defense_ratings, home_advantage):
    standings = {}  # team -> {pts, gf, ga}

    for _, row in group_fixtures.iterrows():
        h, a = row['home_team'], row['away_team']
        mu_h = np.exp(attack_ratings.get(h, 0) - defense_ratings.get(a, 0) + home_advantage)
        mu_a = np.exp(attack_ratings.get(a, 0) - defense_ratings.get(h, 0))

        hg = poisson.rvs(mu_h)  # random scoreline
        ag = poisson.rvs(mu_a)

        for team in [h, a]:
            if team not in standings:
                standings[team] = {'pts': 0, 'gf': 0, 'ga': 0, 'group': row['group']}

        standings[h]['gf'] += hg; standings[h]['ga'] += ag
        standings[a]['gf'] += ag; standings[a]['ga'] += hg

        if hg > ag:
            standings[h]['pts'] += 3
        elif ag > hg:
            standings[a]['pts'] += 3
        else:
            standings[h]['pts'] += 1; standings[a]['pts'] += 1

    return standings

In [ ]:
# Rank each group and extract best 3rd-place teams
def get_group_results(standings, group_fixtures):
    groups = group_fixtures['group'].unique()
    group_winners, group_runners = {}, {}
    third_place_teams = []
    for g in groups:
        teams_in_group = group_fixtures[
            (group_fixtures['group'] == g)
        ][['home_team', 'away_team']].values.ravel()
        teams_in_group = list(pd.unique(teams_in_group))
        ranked = sorted(teams_in_group, key=lambda t: (
            standings.get(t, {}).get('pts', 0),
            standings.get(t, {}).get('gf', 0) - standings.get(t, {}).get('ga', 0),
            standings.get(t, {}).get('gf', 0)
        ), reverse=True)
        group_winners[g]  = ranked[0]
        group_runners[g]  = ranked[1]
        third_place_teams.append({
            'team': ranked[2],
            'group': g,
            'pts': standings.get(ranked[2], {}).get('pts', 0),
            'gd':  standings.get(ranked[2], {}).get('gf', 0) - standings.get(ranked[2], {}).get('ga', 0),
            'gf':  standings.get(ranked[2], {}).get('gf', 0)
        })
    # Best 3rd: top 8 third-place teams by pts, gd, gf
    third_df = pd.DataFrame(third_place_teams).sort_values(
        ['pts', 'gd', 'gf'], ascending=False
    ).reset_index(drop=True)
    best_thirds = third_df['team'].tolist()  # ordered best to worst
    return group_winners, group_runners, best_thirds

In [ ]:
# Monte Carlo: run N simulations, track most frequent outcomes
N = 10000
winner_counts  = {g: {} for g in group_fixtures['group'].unique()}
runner_counts  = {g: {} for g in group_fixtures['group'].unique()}
third_counts   = {}
for _ in range(N):
    standings = simulate_group_stage(group_fixtures, attack_ratings, defense_ratings, home_advantage)
    gw, gr, bt = get_group_results(standings, group_fixtures)
    for g in gw:
        winner_counts[g][gw[g]] = winner_counts[g].get(gw[g], 0) + 1
        runner_counts[g][gr[g]] = runner_counts[g].get(gr[g], 0) + 1
    for t in bt[:8]:
        third_counts[t] = third_counts.get(t, 0) + 1
# Most likely winner/runner per group
predicted_winners = {g: max(winner_counts[g], key=winner_counts[g].get) for g in winner_counts}
predicted_runners = {g: max(runner_counts[g], key=runner_counts[g].get) for g in runner_counts}
predicted_best_thirds = sorted(third_counts, key=third_counts.get, reverse=True)[:8]
print("Winners:", predicted_winners)
print("Runners-up:", predicted_runners)
print("Best 3rd:", predicted_best_thirds[:8])

Winners: {'A': 'Mexico', 'B': 'Canada', 'D': 'USA', 'C': 'Brazil', 'E': 'Ecuador', 'F': 'Netherlands', 'H': 'Spain', 'G': 'Belgium', 'J': 'Argentina', 'I': 'France', 'K': 'Portugal', 'L': 'England'}
Runners-up: {'A': 'Slovenia', 'B': 'Qatar', 'D': 'Australia', 'C': 'Morocco', 'E': 'Germany', 'F': 'Japan', 'H': 'Spain', 'G': 'Egypt', 'J': 'Algeria', 'I': 'Senegal', 'K': 'Portugal', 'L': 'Croatia'}
Best 3rd: ['Uzbekistan', 'Saudi Arabia', 'Haiti', 'Switzerland', 'Paraguay', 'Croatia', 'South Africa', 'Venezuela']


## 🏆 Knockout stage predictions

For knockout matches you also predict **which teams are playing**. Fill in the team names based on your group stage predictions, then add your match predictions.

In [ ]:
knockout_predictions = knockout_slots.copy()

# Fill in your predictions for each knockout match
# Example (match 73 — Round of 32): predicted_home_team='Brazil', predicted_away_team='France', predicted_home_goals=1, predicted_away_goals=0, corners=8, yellow_cards=2, red_cards=0, match_winner='home', penalties=False
knockout_predictions['predicted_home_team']  = None   # e.g. "Brazil"
knockout_predictions['predicted_away_team']  = None   # e.g. "France"
knockout_predictions['predicted_home_goals'] = None   # e.g. 1
knockout_predictions['predicted_away_goals'] = None   # e.g. 0
knockout_predictions['corners']              = None   # e.g. 8
knockout_predictions['yellow_cards']         = None   # e.g. 2
knockout_predictions['red_cards']            = None   # e.g. 0
knockout_predictions['match_winner']         = None   # "home" or "away"
knockout_predictions['penalties']            = None   # True or False

knockout_predictions

,match_id,round,multiplier,date_utc,venue,slot_home,slot_away,predicted_home_team,predicted_away_team,predicted_home_goals,predicted_away_goals,corners,yellow_cards,red_cards,match_winner,penalties
0,73,Round of 32,1,2026-06-28T19:00:00Z,"SoFi Stadium, Los Angeles",Runner-up Group A,Runner-up Group B,None,None,None,None,None,None,None,None,None
1,74,Round of 32,1,2026-06-29T17:00:00Z,"NRG Stadium, Houston",Winner Group C,Runner-up Group F,None,None,None,None,None,None,None,None,None
2,75,Round of 32,1,2026-06-29T20:30:00Z,"Gillette Stadium, Boston",Winner Group E,Best 3rd (Groups A/B/C/D/F),None,None,None,None,None,None,None,None,None
3,76,Round of 32,1,2026-06-30T01:00:00Z,"Estadio BBVA, Monterrey",Winner Group F,Runner-up Group C,None,None,None,None,None,None,None,None,None
4,77,Round of 32,1,2026-06-30T17:00:00Z,"AT&T Stadium, Dallas",Runner-up Group E,Runner-up Group I,None,None,None,None,None,None,None,None,None
5,78,Round of 32,1,2026-06-30T21:00:00Z,"MetLife Stadium, East Rutherford",Winner Group I,Best 3rd (Groups C/D/F/G/H),None,None,None,None,None,None,None,None,None
6,79,Round of 32,1,2026-07-01T01:00:00Z,"Estadio Azteca, Mexico City",Winner Group A,Best 3rd (Groups C/E/F/H/I),None,None,None,None,None,None,None,None,None
7,80,Round of 32,1,2026-07-01T16:00:00Z,"Mercedes-Benz Stadium, Atlanta",Winner Group L,Best 3rd (Groups E/H/I/J/K),None,None,None,None,None,None,None,None,None
8,81,Round of 32,1,2026-07-01T20:00:00Z,"Lumen Field, Seattle",Winner Group G,Best 3rd (Groups A/E/H/I/J),None,None,None,None,None,None,None,None,None
9,82,Round of 32,1,2026-07-02T00:00:00Z,"Levi's Stadium, Santa Clara",Winner Group D,Best 3rd (Groups B/E/F/I/J),None,None,None,None,None,None,None,None,None


Knockout Predictions

In [ ]:
# Map group results to Round of 32 slots
# The slot_home / slot_away columns describe each team slot (e.g. "Winner Group A"). Resolve them to actual teams:

def resolve_slot(slot, predicted_winners, predicted_runners, predicted_best_thirds):
    if slot.startswith("Winner Group "):
        g = slot.split()[-1]
        return predicted_winners.get(g, slot)
    elif slot.startswith("Runner-up Group "):
        g = slot.split()[-1]
        return predicted_runners.get(g, slot)
    elif slot.startswith("Best 3rd"):
        # Return best available 3rd-place team (assign in order)
        return predicted_best_thirds.pop(0) if predicted_best_thirds else slot
    return slot  # fallback for "Winner Match X" — handled in next cell

In [ ]:
# Predict each knockout match and propagate round by round
# Process matches in order — each round's winners feed the next round's slots:

match_winners = {}   # match_id -> winning team name
match_losers = {}    # match_id -> losing team name
match_home    = {}   # match_id -> home team
match_away    = {}   # match_id -> away team
best_thirds_copy = predicted_best_thirds.copy()  # consume in order
for _, row in knockout_slots.iterrows():
    mid = row['match_id']
    # Resolve home team
    sh = row['slot_home']
    if sh.startswith("Winner Match "):
        ref = int(sh.split()[-1])
        home_team = match_winners.get(ref, sh)
    elif sh.startswith("Loser Match "):
        ref = int(sh.split()[-1])
        home_team = match_losers.get(ref, sh)  # for 3rd place playoff
    else:
        home_team = resolve_slot(sh, predicted_winners, predicted_runners, best_thirds_copy)
    # Resolve away team
    sa = row['slot_away']
    if sa.startswith("Winner Match "):
        ref = int(sa.split()[-1])
        away_team = match_winners.get(ref, sa)
    elif sa.startswith("Loser Match "):
        ref = int(sa.split()[-1])
        away_team = match_losers.get(ref, sa)
    else:
        away_team = resolve_slot(sa, predicted_winners, predicted_runners, best_thirds_copy)
    match_home[mid] = home_team
    match_away[mid] = away_team
    # Predict the match
    mu_h, mu_a = expected_goals(home_team, away_team)
    h_goals, a_goals = predict_scoreline(mu_h, mu_a)
    p_home, _, p_away = win_probabilities(mu_h, mu_a)
    winner = home_team if p_home >= p_away else away_team
    loser  = away_team if winner == home_team else home_team
    match_winners[mid] = winner
    match_losers[mid] = loser

In [ ]:
# Fill knockout_predictions

for i, row in knockout_slots.iterrows():
    mid = row['match_id']
    home_team = match_home[mid]
    away_team = match_away[mid]
    mu_h, mu_a = expected_goals(home_team, away_team)
    h_goals, a_goals = predict_scoreline(mu_h, mu_a)
    p_home, _, p_away = win_probabilities(mu_h, mu_a)
    knockout_predictions.loc[i, 'predicted_home_team']  = home_team
    knockout_predictions.loc[i, 'predicted_away_team']  = away_team
    knockout_predictions.loc[i, 'predicted_home_goals'] = h_goals
    knockout_predictions.loc[i, 'predicted_away_goals'] = a_goals
    knockout_predictions.loc[i, 'corners']              = 10
    knockout_predictions.loc[i, 'yellow_cards']         = 3
    knockout_predictions.loc[i, 'red_cards']            = 0
    knockout_predictions.loc[i, 'match_winner']         = 'home' if p_home >= p_away else 'away'
    knockout_predictions.loc[i, 'penalties']            = False  # set below

In [ ]:
knockout_predictions

,match_id,round,multiplier,date_utc,venue,slot_home,slot_away,predicted_home_team,predicted_away_team,predicted_home_goals,predicted_away_goals,corners,yellow_cards,red_cards,match_winner,penalties
0,73,Round of 32,1,2026-06-28T19:00:00Z,"SoFi Stadium, Los Angeles",Runner-up Group A,Runner-up Group B,Slovenia,Qatar,0,1,10,3,0,away,False
1,74,Round of 32,1,2026-06-29T17:00:00Z,"NRG Stadium, Houston",Winner Group C,Runner-up Group F,Brazil,Japan,1,0,10,3,0,home,False
2,75,Round of 32,1,2026-06-29T20:30:00Z,"Gillette Stadium, Boston",Winner Group E,Best 3rd (Groups A/B/C/D/F),Ecuador,Uzbekistan,1,0,10,3,0,home,False
3,76,Round of 32,1,2026-06-30T01:00:00Z,"Estadio BBVA, Monterrey",Winner Group F,Runner-up Group C,Netherlands,Morocco,1,1,10,3,0,away,False
4,77,Round of 32,1,2026-06-30T17:00:00Z,"AT&T Stadium, Dallas",Runner-up Group E,Runner-up Group I,Germany,Senegal,1,0,10,3,0,home,False
5,78,Round of 32,1,2026-06-30T21:00:00Z,"MetLife Stadium, East Rutherford",Winner Group I,Best 3rd (Groups C/D/F/G/H),France,Saudi Arabia,1,0,10,3,0,home,False
6,79,Round of 32,1,2026-07-01T01:00:00Z,"Estadio Azteca, Mexico City",Winner Group A,Best 3rd (Groups C/E/F/H/I),Mexico,Haiti,2,0,10,3,0,home,False
7,80,Round of 32,1,2026-07-01T16:00:00Z,"Mercedes-Benz Stadium, Atlanta",Winner Group L,Best 3rd (Groups E/H/I/J/K),England,Switzerland,2,0,10,3,0,home,False
8,81,Round of 32,1,2026-07-01T20:00:00Z,"Lumen Field, Seattle",Winner Group G,Best 3rd (Groups A/E/H/I/J),Belgium,Paraguay,1,0,10,3,0,home,False
9,82,Round of 32,1,2026-07-02T00:00:00Z,"Levi's Stadium, Santa Clara",Winner Group D,Best 3rd (Groups B/E/F/I/J),USA,Croatia,1,0,10,3,0,home,False


In [ ]:
# Set penalties (base rate ~27%)
# Don't set all to False — that loses easy points. Use the base rate:

# ~27% of knockout matches go to penalties historically
# Favour penalties when teams are closely matched
for i, row in knockout_slots.iterrows():
    mid = row['match_id']
    mu_h, mu_a = expected_goals(match_home[mid], match_away[mid])
    p_home, _, p_away = win_probabilities(mu_h, mu_a)
    strength_diff = abs(p_home - p_away)
    # Close match (diff < 0.15) → predict penalties, otherwise no
    knockout_predictions.loc[i, 'penalties'] = strength_diff < 0.15

 Validate & Clean

Check for remaining None values

In [ ]:
# Group stage — check for None/NaN
print("Group predictions nulls:")
print(group_predictions.isnull().sum())

Group predictions nulls:
match_id                0
group                   0
home_team               0
away_team               0
date_utc                0
venue                   0
predicted_home_goals    0
predicted_away_goals    0
corners                 0
yellow_cards            0
red_cards               0
winning_team            0
dtype: int64


In [ ]:
# Knockout stage — check for None/NaN
print("Knockout predictions nulls:")
print(knockout_predictions.isnull().sum())

Knockout predictions nulls:
match_id                0
round                   0
multiplier              0
date_utc                0
venue                   0
slot_home               0
slot_away               0
predicted_home_team     0
predicted_away_team     0
predicted_home_goals    0
predicted_away_goals    0
corners                 0
yellow_cards            0
red_cards               0
match_winner            0
penalties               0
dtype: int64


Check value constraints

In [ ]:
# winning_team must only be 'home', 'away', or 'draw'
assert set(group_predictions['winning_team'].unique()).issubset({'home', 'away', 'draw'})

# match_winner must only be 'home' or 'away'
assert set(knockout_predictions['match_winner'].unique()).issubset({'home', 'away'})

# penalties must be boolean
assert knockout_predictions['penalties'].dtype == bool or \
       set(knockout_predictions['penalties'].unique()).issubset({True, False})

# Goals must be non-negative integers
assert (group_predictions['predicted_home_goals'] >= 0).all()
assert (group_predictions['predicted_away_goals'] >= 0).all()

print("All checks passed ✓")

All checks passed ✓


Check row counts

In [ ]:
assert len(group_predictions) == 72,    f"Expected 72, got {len(group_predictions)}"
assert len(knockout_predictions) == 32, f"Expected 32, got {len(knockout_predictions)}"
print("Row counts correct ✓")

Row counts correct ✓


## ✅ Checklist before publishing into the competition

- Rename your workspace to make it descriptive of your work. N.B. you should leave the notebook name as `notebook.ipynb`.
- Remove redundant cells like the judging criteria, so the workbook is focused on your predictions.
- Make sure all prediction cells are filled in—`None` values will score 0 points.
- Check that all cells run without error.
- Make sure your workbook is published before **June 10, 2026 at 09:00 UTC**.

## ⏳ Time is ticking. Good luck!

## Verifying model improvements


Step 1 — Create a backtest dataset

In [ ]:
# Test set: past World Cup matches (not used in training)
test = results[
    (results['tournament'] == 'FIFA World Cup') &
    (results['date'] < '2018-01-01')  # matches before your training window
].copy()

test = test.dropna(subset=['home_score', 'away_score'])
test['home_score'] = test['home_score'].astype(int)
test['away_score'] = test['away_score'].astype(int)

Step 2 — Score each prediction using competition rules

In [ ]:
def score_prediction(pred_h, pred_a, actual_h, actual_a):
    points = 0
    # Exact score
    if pred_h == actual_h and pred_a == actual_a:
        return 25
    # Correct goal difference
    if (pred_h - pred_a) == (actual_h - actual_a):
        points = max(points, 10)
    # Correct total goals
    if (pred_h + pred_a) == (actual_h + actual_a):
        points = max(points, 10)
    return points

def score_winner(pred_winner, actual_h, actual_a):
    if actual_h > actual_a:   actual_winner = 'home'
    elif actual_a > actual_h: actual_winner = 'away'
    else:                     actual_winner = 'draw'
    return 40 if pred_winner == actual_winner else 0

Step 3 — Run predictions on the test set and compute average score

In [ ]:
score_results = []

for _, row in test.iterrows():
    h, a = row['home_team'], row['away_team']
    # Skip teams not in the model
    if h not in attack_ratings or a not in attack_ratings:
        continue

    mu_h, mu_a = expected_goals(h, a)
    pred_h, pred_a = predict_scoreline(mu_h, mu_a)
    p_home, p_draw, p_away = win_probabilities(mu_h, mu_a)
    pred_winner = pick_winner(p_home, p_draw, p_away)

    score_results.append({
        'match': f"{h} vs {a}",
        'predicted': f"{pred_h}-{pred_a}",
        'actual': f"{int(row['home_score'])}-{int(row['away_score'])}",
        'score_points': score_prediction(pred_h, pred_a, int(row['home_score']), int(row['away_score'])),
        'winner_points': score_winner(pred_winner, int(row['home_score']), int(row['away_score']))
    })

backtest_df = pd.DataFrame(score_results)
print(backtest_df[['predicted', 'actual', 'score_points', 'winner_points']].describe())
print(f"\nAvg score points per match:  {backtest_df['score_points'].mean():.2f}")
print(f"Avg winner points per match: {backtest_df['winner_points'].mean():.2f}")
print(f"Exact score rate: {(backtest_df['score_points'] == 25).mean():.1%}")
print(f"Correct winner rate: {(backtest_df['winner_points'] == 40).mean():.1%}")

       score_points  winner_points
count    836.000000     836.000000
mean       4.599282      19.425837
std        7.883186      20.003724
min        0.000000       0.000000
25%        0.000000       0.000000
50%        0.000000       0.000000
75%       10.000000      40.000000
max       25.000000      40.000000

Avg score points per match:  4.60
Avg winner points per match: 19.43
Exact score rate: 9.9%
Correct winner rate: 48.6%
